# Tablero de Proyeccion Chain Ladder por Niveles de Riesgo

Proyeccion del triangulo vintage usando **Chain Ladder** para cada uno de los 12 segmentos (CC/PP x Nuevo/Existente x Bajo/Medio/Alto).

Incluye una **proyeccion general** ponderada por cantidad de operaciones al MOB 1 de cada segmento.

---
## 0. Setup

In [ ]:
import os
import subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from matplotlib.lines import Line2D

PROJECT_ROOT = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
SRC_DIR = os.path.join(PROJECT_ROOT, "src")
PROCESSED_DIR = os.path.join(PROJECT_ROOT, "data", "processed")
REPORTS_DIR = os.path.join(PROJECT_ROOT, "reports")

# ---------------------------------------------------------------------------
# PARAMETROS CONFIGURABLES
# ---------------------------------------------------------------------------
MOB_OBJETIVO = 18
N_DESTACAR = 5
COLOR_2024 = "#2171b5"
COLOR_2025 = "#e6550d"
COLOR_2026 = "#888888"
ALPHA_INDIVIDUAL = 0.18
ALPHA_HIGHLIGHT = 0.85

COLOR_NIVEL = {
    "Bajo": "#2ca02c",
    "Medio": "#ff7f0e",
    "Alto": "#d62728",
}

NOMBRE_TIPOOPE = {"CC": "CC", "PP": "PP"}
NOMBRE_CLIENTENUEVO = {"V": "Nuevo", "F": "Existente"}

# ---------------------------------------------------------------------------
# Segmentos excluidos del calculo general ponderado.
# Se proyectan individualmente (monitoreo historico) pero NO participan
# en la proyeccion general porque actualmente no se vende a estos segmentos.
# DEBE coincidir con SEGMENTOS_EXCLUIDOS_GENERAL del script.
# ---------------------------------------------------------------------------
SEGMENTOS_EXCLUIDOS_GENERAL = [
    "CC_V_Medio",
    "CC_V_Alto",
    "CC_F_Alto",
    "PP_V_Medio",
    "PP_V_Alto",
    "PP_F_Medio",
    "PP_F_Alto",
]

print(f"Raiz del proyecto: {PROJECT_ROOT}")
print(f"MOB objetivo: {MOB_OBJETIVO}")
print(f"Segmentos excluidos del general: {SEGMENTOS_EXCLUIDOS_GENERAL}")

---
## 1. Ejecutar pipeline (consolidacion + matrices + proyeccion CL)

In [ ]:
scripts = [
    "consolidar_vintage_niveles.py",
    "generar_matriz_vintage_niveles.py",
    "generar_proyeccion_chainladder_niveles.py",
]
for script in scripts:
    print(f"--- Ejecutando {script} ---")
    result = subprocess.run(
        ["py", os.path.join(SRC_DIR, script)],
        capture_output=True, text=True, cwd=PROJECT_ROOT
    )
    print(result.stdout[-800:] if len(result.stdout) > 800 else result.stdout)
    if result.returncode != 0:
        print("ERROR:", result.stderr)
        break

In [ ]:
# Leer datos generados
matrices_obs = pd.read_csv(os.path.join(PROCESSED_DIR, "matrices_vintage_niveles.csv"), sep=";", decimal=",")
matrices_proy = pd.read_csv(os.path.join(PROCESSED_DIR, "matrices_proyectadas_niveles.csv"), sep=";", decimal=",")
marcadores_df = pd.read_csv(os.path.join(PROCESSED_DIR, "marcadores_proyectados_niveles.csv"), sep=";")
factores_df = pd.read_csv(os.path.join(PROCESSED_DIR, "factores_cl_niveles.csv"), sep=";", decimal=",")
general_proy = pd.read_csv(os.path.join(PROCESSED_DIR, "proyeccion_general_niveles.csv"), sep=";", decimal=",", index_col=0)
df_consolidado = pd.read_csv(os.path.join(PROCESSED_DIR, "vintage_niveles_consolidado.csv"), sep=";", decimal=",")

segmentos = sorted(matrices_proy["segmento"].unique())
print(f"Segmentos: {len(segmentos)}")
print(f"Cohortes proyeccion general: {len(general_proy)}")

---
## 2. Factores Chain Ladder por segmento

Comparacion de factores CL entre los 3 niveles de riesgo para cada combinacion tipoope x clientenuevo.

In [ ]:
combos = [("CC", "F"), ("CC", "V"), ("PP", "F"), ("PP", "V")]
niveles = ["Bajo", "Medio", "Alto"]

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle("Factores Chain Ladder por Nivel de Riesgo",
             fontsize=14, fontweight="bold", y=0.98)

for idx, (tipoope, clientenuevo) in enumerate(combos):
    ax = axes[idx // 2, idx % 2]
    tipo_n = NOMBRE_TIPOOPE[tipoope]
    cliente_n = NOMBRE_CLIENTENUEVO[clientenuevo]

    for nivel in niveles:
        seg = f"{tipoope}_{clientenuevo}_{nivel}"
        row = factores_df[factores_df["segmento"] == seg]
        if row.empty:
            continue
        cols_trans = [c for c in row.columns if "->" in c]
        vals = row[cols_trans].iloc[0].values.astype(float)
        mob_destino = [int(c.split("->")[1]) for c in cols_trans]
        ax.plot(mob_destino, vals, color=COLOR_NIVEL[nivel],
                linewidth=2, marker="o", markersize=4, label=nivel)

    ax.axhline(y=1.0, color="#333", linewidth=1, linestyle=":", alpha=0.7)
    ax.set_title(f"{tipo_n} - {cliente_n}", fontsize=12, fontweight="bold")
    ax.set_xlabel("MOB destino", fontsize=9)
    ax.set_ylabel("Factor CL", fontsize=9)
    ax.grid(True, alpha=0.25, linestyle="--")
    ax.legend(fontsize=9)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig(os.path.join(REPORTS_DIR, "factores_cl_niveles.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Guardado en reports/factores_cl_niveles.png")

---
## 3. Curvas Vintage Proyectadas por Segmento (12 graficos)

Lineas **solidas** = observado. Lineas **punteadas** = proyeccion Chain Ladder.

In [ ]:
LW_INDIVIDUAL = 1.0
LW_HIGHLIGHT = 2.2
COLORES_DESTACADAS = ["#d73027", "#fc8d59", "#fee08b", "#91cf60", "#1a9850"]

def get_matriz(df, segmento):
    sub = df[df["segmento"] == segmento].copy()
    mob_cols = [c for c in sub.columns if c.startswith("MOB_")]
    return sub.set_index("cohorte")[mob_cols]

def extraer_serie(matriz, cohorte):
    vals = matriz.loc[cohorte].dropna()
    mobs = [int(c.replace("MOB_", "")) for c in vals.index]
    return mobs, vals.values

def parte_proyectada(m_obs, m_proy, cohorte):
    obs_mobs, _ = extraer_serie(m_obs, cohorte)
    proy_mobs, proy_vals = extraer_serie(m_proy, cohorte)
    ultimo_obs = max(obs_mobs)
    mask = [m >= ultimo_obs for m in proy_mobs]
    return ([m for m, k in zip(proy_mobs, mask) if k],
            [v for v, k in zip(proy_vals, mask) if k])

fig, axes = plt.subplots(4, 3, figsize=(22, 24))
fig.suptitle(f"Curvas Vintage Proyectadas (CL hasta MOB {MOB_OBJETIVO}) por Segmento",
             fontsize=16, fontweight="bold", y=0.98)

for idx, segmento in enumerate(segmentos):
    ax = axes[idx // 3, idx % 3]
    m_obs = get_matriz(matrices_obs, segmento)
    m_proy = get_matriz(matrices_proy, segmento)

    cohortes_ord = sorted(m_proy.index)
    ultimas_n = cohortes_ord[-N_DESTACAR:]
    color_dest = {c: COLORES_DESTACADAS[i % len(COLORES_DESTACADAS)]
                  for i, c in enumerate(ultimas_n)}

    for cohorte in cohortes_ord:
        es_dest = cohorte in ultimas_n
        if cohorte.startswith("2024"):
            color_base = COLOR_2024
        elif cohorte.startswith("2025"):
            color_base = COLOR_2025
        else:
            color_base = COLOR_2026
        color = color_dest[cohorte] if es_dest else color_base
        alpha = ALPHA_HIGHLIGHT if es_dest else ALPHA_INDIVIDUAL
        lw = LW_HIGHLIGHT if es_dest else LW_INDIVIDUAL

        # Observado (solida)
        obs_m, obs_v = extraer_serie(m_obs, cohorte)
        ax.plot(obs_m, obs_v, color=color, alpha=alpha, linewidth=lw,
                linestyle="-", zorder=3 if es_dest else 1)

        # Proyectado (punteada)
        proy_m, proy_v = parte_proyectada(m_obs, m_proy, cohorte)
        if len(proy_m) > 1:
            ax.plot(proy_m, proy_v, color=color, alpha=alpha * 0.7,
                    linewidth=lw, linestyle="--", zorder=2 if es_dest else 0)

    # Etiquetas destacadas
    for cohorte in ultimas_n:
        if cohorte in m_proy.index:
            proy_m, proy_v = extraer_serie(m_proy, cohorte)
            ax.annotate(cohorte, xy=(proy_m[-1], proy_v[-1]),
                        xytext=(4, 0), textcoords="offset points",
                        fontsize=7, fontweight="bold",
                        color=color_dest[cohorte], va="center")

    partes = segmento.split("_")
    tipo = NOMBRE_TIPOOPE.get(partes[0], partes[0])
    cliente = NOMBRE_CLIENTENUEVO.get(partes[1], partes[1])
    nivel = partes[2]
    ax.set_title(f"{tipo} - {cliente} - {nivel}",
                 fontsize=11, fontweight="bold", color=COLOR_NIVEL.get(nivel, "black"))
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1, decimals=0))
    ax.set_xticks(range(1, MOB_OBJETIVO + 1))
    ax.set_xlim(0.5, MOB_OBJETIVO + 1.5)
    ax.grid(True, alpha=0.25, linestyle="--")
    ax.set_xlabel("MOB", fontsize=9)
    ax.set_ylabel("Indice Morosidad", fontsize=9)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig(os.path.join(REPORTS_DIR, "curvas_proyeccion_niveles_12.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Guardado en reports/curvas_proyeccion_niveles_12.png")

---
## 4. Comparacion de Ultimates por Nivel

Indice de morosidad proyectado al MOB objetivo, comparando niveles dentro de cada combinacion tipoope x clientenuevo.

In [ ]:
col_ult = f"MOB_{MOB_OBJETIVO}"

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle(f"Ultimate (MOB {MOB_OBJETIVO}) por Nivel de Riesgo",
             fontsize=14, fontweight="bold", y=0.98)

for idx, (tipoope, clientenuevo) in enumerate(combos):
    ax = axes[idx // 2, idx % 2]
    tipo_n = NOMBRE_TIPOOPE[tipoope]
    cliente_n = NOMBRE_CLIENTENUEVO[clientenuevo]

    for nivel in niveles:
        seg = f"{tipoope}_{clientenuevo}_{nivel}"
        m_proy = get_matriz(matrices_proy, seg)
        if col_ult not in m_proy.columns:
            continue
        ultimates = m_proy[col_ult].dropna()
        ax.barh([f"{c}" for c in ultimates.index],
                ultimates.values,
                color=COLOR_NIVEL[nivel], alpha=0.6,
                label=f"{nivel} (prom={ultimates.mean():.1%})")

    ax.set_title(f"{tipo_n} - {cliente_n}", fontsize=12, fontweight="bold")
    ax.xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1, decimals=0))
    ax.grid(True, alpha=0.25, linestyle="--", axis="x")
    ax.legend(fontsize=9, loc="lower right")
    ax.invert_yaxis()
    ax.set_xlabel(f"Indice al MOB {MOB_OBJETIVO}", fontsize=9)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig(os.path.join(REPORTS_DIR, "ultimates_niveles.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Guardado en reports/ultimates_niveles.png")

---
## 5. Tabla de Mora Ultima por Segmento

In [ ]:
col_ult = f"MOB_{MOB_OBJETIVO}"

resumen_all = []
for segmento in segmentos:
    m_obs = get_matriz(matrices_obs, segmento)
    m_proy = get_matriz(matrices_proy, segmento)

    for cohorte in sorted(m_proy.index):
        obs = m_obs.loc[cohorte].dropna()
        ultimo_mob_obs = max(int(c.replace("MOB_", "")) for c in obs.index)
        indice_actual = obs.iloc[-1]
        indice_ult = m_proy.loc[cohorte, col_ult] if col_ult in m_proy.columns else np.nan
        resumen_all.append({
            "segmento": segmento,
            "cohorte": cohorte,
            "mob_actual": ultimo_mob_obs,
            "indice_actual": indice_actual,
            "ultimate": indice_ult,
            "desarrollo_restante": indice_ult - indice_actual if pd.notna(indice_ult) else np.nan,
            "mobs_proyectados": max(0, MOB_OBJETIVO - ultimo_mob_obs),
        })

df_ult = pd.DataFrame(resumen_all)

# Resumen agregado por segmento
resumen_seg = df_ult.groupby("segmento").agg(
    cohortes=("cohorte", "count"),
    ult_promedio=("ultimate", "mean"),
    ult_min=("ultimate", "min"),
    ult_max=("ultimate", "max"),
    cohorte_peor=("ultimate", "idxmax"),
).sort_values("ult_promedio", ascending=False)

# Reemplazar idx por cohorte real
resumen_seg["cohorte_peor"] = resumen_seg["cohorte_peor"].apply(
    lambda i: df_ult.loc[i, "cohorte"] if pd.notna(i) else "-")

print(f"MORA ULTIMA (MOB {MOB_OBJETIVO}) - PROMEDIO POR SEGMENTO")
print("=" * 75)
print(f"{'Segmento':>20s} | {'n':>3s} | {'Promedio':>9s} | {'Min':>9s} | {'Max':>9s} | {'Peor cohorte':>12s}")
print("-" * 75)
for seg, row in resumen_seg.iterrows():
    print(f"{seg:>20s} | {int(row['cohortes']):>3d} | {row['ult_promedio']:>8.2%} | "
          f"{row['ult_min']:>8.2%} | {row['ult_max']:>8.2%} | {row['cohorte_peor']:>12s}")
print("=" * 75)

---
## 6. Proyeccion General (promedio ponderado por operaciones)

Curva general que pondera los 12 segmentos por su cantidad de operaciones al MOB 1.
Los segmentos con mas operaciones tienen mas peso en la curva general.

In [ ]:
# Mostrar pesos
mob1 = df_consolidado[df_consolidado["mob"] == 1]
pesos_tot = mob1.groupby("segmento")["cantidad_operaciones"].sum().sort_values(ascending=False)
total_ops = pesos_tot.sum()

# Pesos solo de segmentos activos
pesos_activos = pesos_tot.drop(SEGMENTOS_EXCLUIDOS_GENERAL, errors="ignore")
total_activos = pesos_activos.sum()

print("PESOS DE PONDERACION (operaciones al MOB 1):")
print("=" * 65)
for seg, ops in pesos_tot.items():
    excluido = " [EXCLUIDO]" if seg in SEGMENTOS_EXCLUIDOS_GENERAL else ""
    pct_activo = f"({ops/total_activos*100:>5.1f}% del gral)" if seg not in SEGMENTOS_EXCLUIDOS_GENERAL else ""
    barra = '#' * int(ops / total_ops * 50)
    print(f"  {seg:20s}: {ops:>9,.0f} ({ops/total_ops*100:>5.1f}%) {barra}{excluido} {pct_activo}")
print(f"  {'TOTAL':20s}: {total_ops:>9,.0f}")
print(f"  {'TOTAL ACTIVOS':20s}: {total_activos:>9,.0f} ({total_activos/total_ops*100:.1f}%)")
print("=" * 65)

In [ ]:
# Construir ultimo MOB observado por cohorte (solo segmentos activos)
matrices_obs_dict = {}
for seg in segmentos:
    matrices_obs_dict[seg] = get_matriz(matrices_obs, seg)

mob1_pivot = mob1.pivot_table(index="cohorte", columns="segmento",
                              values="cantidad_operaciones", aggfunc="sum")

# Segmentos activos para el calculo general
segmentos_activos = [s for s in segmentos if s not in SEGMENTOS_EXCLUIDOS_GENERAL]

ultimo_mob_obs_general = {}
for cohorte in general_proy.index:
    mob_maxs = []
    for seg in segmentos_activos:
        if seg not in matrices_obs_dict:
            continue
        m = matrices_obs_dict[seg]
        if cohorte not in m.index:
            continue
        peso = mob1_pivot.loc[cohorte, seg] if (cohorte in mob1_pivot.index and seg in mob1_pivot.columns
                                                 and pd.notna(mob1_pivot.loc[cohorte, seg])) else 0
        if peso > 0:
            obs_vals = m.loc[cohorte].dropna()
            if len(obs_vals) > 0:
                mob_maxs.append(max(int(c.replace("MOB_", "")) for c in obs_vals.index))
    ultimo_mob_obs_general[cohorte] = min(mob_maxs) if mob_maxs else 0

# Grafico curvas general
cohortes_ord = sorted(general_proy.index)
ultimas_n = cohortes_ord[-N_DESTACAR:]

COLORES_DESTACADAS = ["#d73027", "#fc8d59", "#fee08b", "#91cf60", "#1a9850"]
color_dest = {c: COLORES_DESTACADAS[i % len(COLORES_DESTACADAS)]
              for i, c in enumerate(ultimas_n)}

fig, ax = plt.subplots(figsize=(15, 7))

for cohorte in cohortes_ord:
    vals = general_proy.loc[cohorte].dropna()
    mobs = [int(c.replace("MOB_", "")) for c in vals.index]
    values = vals.values

    es_dest = cohorte in ultimas_n
    if cohorte.startswith("2024"):
        color_base = COLOR_2024
    elif cohorte.startswith("2025"):
        color_base = COLOR_2025
    else:
        color_base = COLOR_2026
    color = color_dest[cohorte] if es_dest else color_base
    alpha = ALPHA_HIGHLIGHT if es_dest else ALPHA_INDIVIDUAL
    lw = 2.2 if es_dest else 1.0

    ult_obs = ultimo_mob_obs_general.get(cohorte, 0)

    # Observado (solida)
    mask_obs = [m <= ult_obs for m in mobs]
    mobs_obs = [m for m, k in zip(mobs, mask_obs) if k]
    vals_obs = [v for v, k in zip(values, mask_obs) if k]
    if mobs_obs:
        ax.plot(mobs_obs, vals_obs, color=color, alpha=alpha, linewidth=lw,
                linestyle="-", zorder=3 if es_dest else 1)

    # Proyectado (punteada)
    mask_proy = [m >= ult_obs for m in mobs]
    mobs_p = [m for m, k in zip(mobs, mask_proy) if k]
    vals_p = [v for v, k in zip(values, mask_proy) if k]
    if len(mobs_p) > 1:
        ax.plot(mobs_p, vals_p, color=color, alpha=alpha * 0.7, linewidth=lw,
                linestyle="--", zorder=2 if es_dest else 0)

# Etiquetas
for cohorte in ultimas_n:
    vals = general_proy.loc[cohorte].dropna()
    if len(vals) > 0:
        mobs = [int(c.replace("MOB_", "")) for c in vals.index]
        ax.annotate(cohorte, xy=(mobs[-1], vals.iloc[-1]),
                    xytext=(6, 0), textcoords="offset points",
                    fontsize=9, fontweight="bold",
                    color=color_dest[cohorte], va="center", zorder=5)
        ax.plot(mobs[-1], vals.iloc[-1], "o",
                color=color_dest[cohorte], markersize=6, zorder=5)

ax.set_xlabel("Months on Books (MOB)", fontsize=12)
ax.set_ylabel("Indice de Morosidad", fontsize=12)
ax.set_title(f"Proyeccion General Ponderada (CL hasta MOB {MOB_OBJETIVO}) — solo segmentos activos",
             fontsize=14, fontweight="bold")
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1, decimals=0))
ax.set_xticks(range(1, MOB_OBJETIVO + 1))
ax.set_xlim(0.5, MOB_OBJETIVO + 1.5)
ax.grid(True, alpha=0.25, linestyle="--")

legend_elements = [
    Line2D([0], [0], color=COLOR_2024, alpha=ALPHA_INDIVIDUAL, lw=4, label="Cohortes 2024"),
    Line2D([0], [0], color=COLOR_2025, alpha=ALPHA_INDIVIDUAL, lw=4, label="Cohortes 2025"),
    Line2D([0], [0], color="black", lw=1.5, ls="-", label="Observado"),
    Line2D([0], [0], color="black", lw=1.5, ls="--", alpha=0.7, label="Proyectado (CL)"),
]
for cohorte in ultimas_n:
    legend_elements.append(
        Line2D([0], [0], color=color_dest[cohorte], lw=2.2,
               alpha=ALPHA_HIGHLIGHT, label=cohorte))

ax.legend(handles=legend_elements, loc="upper left", fontsize=9, framealpha=0.9)

plt.tight_layout()
plt.savefig(os.path.join(REPORTS_DIR, "proyeccion_general_niveles.png"), dpi=150, bbox_inches="tight")
plt.show()
print(f"Guardado en reports/proyeccion_general_niveles.png")
print(f"Nota: general calculada solo con {len(segmentos_activos)} segmentos activos (excluidos: {len(SEGMENTOS_EXCLUIDOS_GENERAL)})")

### Detalle de las ultimas 5 cohortes (proyeccion general ponderada)

Tabla MOB a MOB mostrando el indice de morosidad observado y proyectado. Los valores **proyectados** se marcan en *italica*.

In [ ]:
# Tabla detallada de las ultimas N cohortes - proyeccion general ponderada
ultimas_cohortes = sorted(general_proy.index)[-N_DESTACAR:]
mob_cols = [f"MOB_{m}" for m in range(1, MOB_OBJETIVO + 1)]

# Armar tabla con valores y marcador obs/proy
detalle = general_proy.loc[ultimas_cohortes, mob_cols].copy()

# Determinar que es observado y que proyectado
def estilo_proy(row):
    cohorte = row.name
    ult_obs = ultimo_mob_obs_general.get(cohorte, 0)
    estilos = []
    for col in row.index:
        mob_num = int(col.replace("MOB_", ""))
        if mob_num > ult_obs:
            estilos.append("background-color: #f0f0f0; font-style: italic; color: #666")
        else:
            estilos.append("")
    return estilos

# Mostrar tabla impresa
print(f"DETALLE MOB A MOB - ULTIMAS {N_DESTACAR} COHORTES (Proyeccion General Ponderada)")
print("=" * 120)
header = f"{'Cohorte':>10s} | {'MOB obs':>7s} |"
for m in range(1, MOB_OBJETIVO + 1):
    header += f" {'MOB '+str(m):>7s}"
print(header)
print("-" * 120)

for cohorte in ultimas_cohortes:
    ult_obs = ultimo_mob_obs_general.get(cohorte, 0)
    line = f"{cohorte:>10s} | {ult_obs:>7d} |"
    for m in range(1, MOB_OBJETIVO + 1):
        col = f"MOB_{m}"
        val = detalle.loc[cohorte, col]
        if pd.isna(val):
            line += f" {'---':>7s}"
        elif m > ult_obs:
            line += f" {val:>6.2%}*"
        else:
            line += f" {val:>7.2%}"
    print(line)

print("-" * 120)
print("* = valor proyectado por Chain Ladder")
print("=" * 120)

# Tabla estilizada
detalle.style.format("{:.2%}", na_rep="-").apply(estilo_proy, axis=1).set_caption(
    f"Proyeccion General Ponderada - Ultimas {N_DESTACAR} cohortes (gris = proyectado)"
)

---
## 7. Comparacion Ultimates: General vs por Nivel

Curvas promedio al MOB objetivo comparando la general vs cada nivel.

In [ ]:
col_ult = f"MOB_{MOB_OBJETIVO}"

# Promedio por nivel (solo segmentos activos)
ult_por_nivel = {}
for nivel in niveles:
    segs_nivel = [s for s in segmentos if s.endswith(f"_{nivel}") and s not in SEGMENTOS_EXCLUIDOS_GENERAL]
    vals = []
    for seg in segs_nivel:
        m = get_matriz(matrices_proy, seg)
        if col_ult in m.columns:
            vals.extend(m[col_ult].dropna().tolist())
    if vals:
        ult_por_nivel[nivel] = vals

# General
ult_general = general_proy[col_ult].dropna().values

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Box plot por nivel + general
data_box = []
labels_box = []
colors_box = []
for nivel in niveles:
    if nivel in ult_por_nivel:
        data_box.append(ult_por_nivel[nivel])
        labels_box.append(nivel)
        colors_box.append(COLOR_NIVEL[nivel])
data_box.append(ult_general)
labels_box.append("General\n(ponderado)")
colors_box.append("#555555")

bp = ax1.boxplot(data_box, labels=labels_box, patch_artist=True, widths=0.5)
for patch, color in zip(bp["boxes"], colors_box):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)
ax1.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1, decimals=0))
ax1.set_title(f"Distribucion de Ultimates (MOB {MOB_OBJETIVO}) — solo activos",
              fontsize=13, fontweight="bold")
ax1.set_ylabel("Indice de Morosidad")
ax1.grid(True, alpha=0.25, linestyle="--", axis="y")

# Barras promedio
promedios = [np.mean(d) for d in data_box]
ax2.barh(labels_box, promedios, color=colors_box, alpha=0.7, edgecolor="white")
for i, v in enumerate(promedios):
    ax2.text(v + 0.002, i, f"{v:.1%}", va="center", fontsize=10, fontweight="bold")
ax2.xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1, decimals=0))
ax2.set_title(f"Promedio Ultimate (MOB {MOB_OBJETIVO}) — solo activos",
              fontsize=13, fontweight="bold")
ax2.grid(True, alpha=0.25, linestyle="--", axis="x")

plt.tight_layout()
plt.savefig(os.path.join(REPORTS_DIR, "comparacion_ultimates_niveles.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Guardado en reports/comparacion_ultimates_niveles.png")

---
## 8. Resumen Ejecutivo

In [ ]:
col_ult = f"MOB_{MOB_OBJETIVO}"

print("=" * 70)
print(f"RESUMEN EJECUTIVO - PROYECCION CHAIN LADDER POR NIVELES (MOB {MOB_OBJETIVO})")
print("=" * 70)

# General
ult_gen = general_proy[col_ult].dropna()
print(f"\nPROYECCION GENERAL (ponderada, solo segmentos activos):")
print(f"  Segmentos activos: {len(segmentos_activos)} de {len(segmentos)}")
print(f"  Promedio:  {ult_gen.mean():.2%}")
print(f"  Minimo:    {ult_gen.min():.2%} ({ult_gen.idxmin()})")
print(f"  Maximo:    {ult_gen.max():.2%} ({ult_gen.idxmax()})")

# Por nivel (solo activos)
print(f"\nPOR NIVEL DE RIESGO (solo segmentos activos):")
print("-" * 50)
for nivel in niveles:
    segs = [s for s in segmentos if s.endswith(f"_{nivel}") and s not in SEGMENTOS_EXCLUIDOS_GENERAL]
    vals = []
    for seg in segs:
        m = get_matriz(matrices_proy, seg)
        if col_ult in m.columns:
            vals.extend(m[col_ult].dropna().tolist())
    if vals:
        print(f"  {nivel:6s}: promedio={np.mean(vals):.2%}, "
              f"min={np.min(vals):.2%}, max={np.max(vals):.2%}, n={len(vals)} ({segs})")

# Por segmento (todos, marcando excluidos)
print(f"\nPOR SEGMENTO (ordenado por ultimate promedio):")
print("-" * 75)
seg_ults = []
for seg in segmentos:
    m = get_matriz(matrices_proy, seg)
    if col_ult in m.columns:
        ult_vals = m[col_ult].dropna()
        if len(ult_vals) > 0:
            seg_ults.append((seg, ult_vals.mean(), ult_vals.min(), ult_vals.max(), len(ult_vals)))

seg_ults.sort(key=lambda x: x[1], reverse=True)
for seg, prom, mn, mx, n in seg_ults:
    partes = seg.split("_")
    tipo = NOMBRE_TIPOOPE.get(partes[0], partes[0])
    cliente = NOMBRE_CLIENTENUEVO.get(partes[1], partes[1])
    nivel = partes[2]
    nombre = f"{tipo} {cliente} {nivel}"
    excl = " [EXCL]" if seg in SEGMENTOS_EXCLUIDOS_GENERAL else ""
    print(f"  {nombre:25s}: {prom:>7.2%} (rango [{mn:.2%}, {mx:.2%}], n={n}){excl}")

# Spread Alto vs Bajo (solo activos)
print(f"\nSPREAD ALTO - BAJO (solo activos):")
print("-" * 50)
for tipoope, clientenuevo in combos:
    tipo_n = NOMBRE_TIPOOPE[tipoope]
    cliente_n = NOMBRE_CLIENTENUEVO[clientenuevo]
    ults = {}
    for nivel in niveles:
        seg = f"{tipoope}_{clientenuevo}_{nivel}"
        if seg in SEGMENTOS_EXCLUIDOS_GENERAL:
            continue
        m = get_matriz(matrices_proy, seg)
        if col_ult in m.columns:
            v = m[col_ult].dropna()
            if len(v) > 0:
                ults[nivel] = v.mean()
    if len(ults) >= 2:
        niveles_disp = sorted(ults.keys(), key=lambda x: ults[x])
        peor = niveles_disp[-1]
        mejor = niveles_disp[0]
        spread = ults[peor] - ults[mejor]
        print(f"  {tipo_n} {cliente_n}: {peor}={ults[peor]:.2%} vs {mejor}={ults[mejor]:.2%} "
              f"-> spread={spread:+.2%}")
    elif len(ults) == 1:
        nivel_unico = list(ults.keys())[0]
        print(f"  {tipo_n} {cliente_n}: solo {nivel_unico}={ults[nivel_unico]:.2%} activo")
    else:
        print(f"  {tipo_n} {cliente_n}: sin segmentos activos")

print("\n" + "=" * 70)